In [ ]:
# ─────────────────────────────────────────────
# CELL 1 — Dependencies
# ─────────────────────────────────────────────
import sys
!{sys.executable} -m pip install -q \
    "diffusers>=0.25.0" transformers accelerate xformers \
    scikit-image lpips clean-fid opencv-python-headless \
    huggingface_hub
print("Dependencies ready")

In [ ]:
# ─────────────────────────────────────────────
# CELL 2 — Config & Paths
# ─────────────────────────────────────────────
import random
from pathlib import Path
import numpy as np
import pandas as pd
import torch

SEED = 42
random.seed(SEED); np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

# ── Dataset (Kaggle) ───────────────────────────────────────────────────────
DATASET_ROOT = Path("/kaggle/input/datasets/dangvy1507/occlusion")
SYNTH_DIR    = DATASET_ROOT / "synthetic_occ"
GT_DIR       = SYNTH_DIR / "x_gt"
OCC_DIR      = SYNTH_DIR / "x_occ"
MASK_DIR     = SYNTH_DIR / "masks"
META_CSV     = SYNTH_DIR / "metadata_synthetic_occ.csv"

# ── Output ─────────────────────────────────────────────────────────────────
OUT_BASE      = Path("/kaggle/working/eval_cn_ipa")
PRED_DIR      = OUT_BASE / "x_hat"
EVAL_GT_DIR   = OUT_BASE / "fid_gt"
EVAL_PRED_DIR = OUT_BASE / "fid_pred"
REPORT_DIR    = OUT_BASE / "reports"
for d in [PRED_DIR, EVAL_GT_DIR, EVAL_PRED_DIR, REPORT_DIR]:
    d.mkdir(parents=True, exist_ok=True)

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
DTYPE  = torch.float16 if DEVICE == "cuda" else torch.float32

# ── Model IDs ──────────────────────────────────────────────────────────────
SD_MODEL_ID = "runwayml/stable-diffusion-inpainting"
CN_MODEL_ID = "lllyasviel/control_v11p_sd15_canny"
IP_REPO     = "h94/IP-Adapter"
IP_WEIGHT   = "models/ip-adapter-plus_sd15.bin"

# ── Inference params ───────────────────────────────────────────────────────
PROMPT     = "a car, realistic, high quality, detailed, complete, no occlusion"
NEG_PROMPT = "blurry, distorted, artifacts, extra car, duplicate"
NUM_STEPS  = 20
GUIDANCE   = 7.5

# ── Evaluation config ──────────────────────────────────────────────────────
TEST_SIZE   = 2000      # tổng số ảnh test
GRID_SAMPLE = 40        # số ảnh dùng cho grid search

# ── Grid search ranges ─────────────────────────────────────────────────────
CN_SCALE_GRID = [0.15, 0.30, 0.45, 0.60, 0.75]
IP_SCALE_GRID = [0.3, 0.5, 0.7]

# ── Canny params ───────────────────────────────────────────────────────────
CANNY_LOW      = 80
CANNY_HIGH     = 150
MASK_DILATE_PX = 5

# ── Bins (occlusion level ablation) ───────────────────────────────────────
BIN_EDGES = [(0.20, 0.40), (0.40, 0.60), (0.60, 0.80)]
BIN_LABELS = ["20-40%", "40-60%", "60-80%"]

meta = pd.read_csv(META_CSV)
print(f"Dataset    : {len(meta):,} images")
print(f"Device     : {DEVICE} | dtype={DTYPE}")
print(f"SD model   : {SD_MODEL_ID}")
print(f"ControlNet : {CN_MODEL_ID}")
print(f"IP-Adapter : {IP_REPO}/{IP_WEIGHT}")
print(f"Test size  : {TEST_SIZE:,}")

In [ ]:
# ─────────────────────────────────────────────
# CELL 3 — Load ControlNet + SD Inpainting + IP-Adapter
# ─────────────────────────────────────────────
import cv2
from PIL import Image
from diffusers import (
    StableDiffusionControlNetInpaintPipeline,
    ControlNetModel,
    DPMSolverMultistepScheduler,
)

print("Loading ControlNet ...")
controlnet = ControlNetModel.from_pretrained(CN_MODEL_ID, torch_dtype=DTYPE)

print("Loading SD1.5 Inpainting pipeline ...")
pipe = StableDiffusionControlNetInpaintPipeline.from_pretrained(
    SD_MODEL_ID,
    controlnet=controlnet,
    torch_dtype=DTYPE,
    safety_checker=None,
    requires_safety_checker=False,
)
pipe.scheduler = DPMSolverMultistepScheduler.from_config(pipe.scheduler.config)
pipe = pipe.to(DEVICE)

# ── xFormers / attention slicing ──────────────────────────────────────────
xformers_ok = False
try:
    pipe.enable_xformers_memory_efficient_attention()
    xformers_ok = True
    print("xFormers: enabled")
except Exception:
    pipe.enable_attention_slicing("auto")
    print("xFormers: fallback → attention slicing")
pipe.vae.enable_slicing()

# ── IP-Adapter ────────────────────────────────────────────────────────────
use_ip = False
try:
    pipe.load_ip_adapter(IP_REPO, subfolder="models", weight_name="ip-adapter-plus_sd15.bin")
    use_ip = True
    print("IP-Adapter: loaded OK")
except Exception as e:
    print(f"IP-Adapter: FAILED to load ({e}) — sẽ chạy không có IP-Adapter")

print(f"\nPipeline ready | IP-Adapter={use_ip} | xFormers={xformers_ok}")

# VRAM baseline
if torch.cuda.is_available():
    props = torch.cuda.get_device_properties(0)
    alloc = torch.cuda.memory_allocated(0) / 1024**3
    total = props.total_memory / 1024**3
    print(f"GPU: {props.name} | Total: {total:.1f} GiB | Alloc after load: {alloc:.2f} GiB")

In [ ]:
# ─────────────────────────────────────────────
# CELL 4 — Helpers: Canny, visible patch, inference wrapper
# ─────────────────────────────────────────────

def extract_canny_masked(
    image_pil: Image.Image,
    mask_pil : Image.Image,
    low  : int = CANNY_LOW,
    high : int = CANNY_HIGH,
    dilate_px: int = MASK_DILATE_PX,
) -> Image.Image:
    """
    Canny từ x_occ, xoá cạnh trong vùng mask để ControlNet
    không học cấu trúc của vật che — chỉ học cạnh xe còn thấy.
    """
    img   = np.array(image_pil.convert("RGB"))
    gray  = cv2.cvtColor(img, cv2.COLOR_RGB2GRAY)
    edges = cv2.Canny(gray, low, high)

    mask_np = np.array(mask_pil.convert("L"))
    if dilate_px > 0:
        k = cv2.getStructuringElement(
            cv2.MORPH_ELLIPSE, (dilate_px * 2 + 1, dilate_px * 2 + 1))
        mask_np = cv2.dilate(mask_np, k, iterations=1)
    edges[mask_np > 127] = 0

    return Image.fromarray(np.stack([edges] * 3, axis=2))


def extract_visible_patch(
    image_pil: Image.Image,
    mask_pil : Image.Image,
) -> Image.Image:
    """
    Patch màu của phần xe còn nhìn thấy (không bị che).
    Dùng làm image-prompt cho IP-Adapter để giữ màu sắc nhất quán.
    """
    img  = np.array(image_pil.convert("RGB"))
    mask = np.array(mask_pil.convert("L"))
    vis  = img.copy()
    vis[mask > 127] = 128   # neutral gray ở vùng bị che
    return Image.fromarray(vis)


def inpaint(
    image   : Image.Image,
    mask    : Image.Image,
    prompt  : str,
    cn_scale: float = 0.45,
    ip_scale: float = 0.5,
    seed    : int   = SEED,
) -> Image.Image:
    """
    SD1.5 Inpainting + ControlNet Canny + IP-Adapter (nếu có).
    """
    canny   = extract_canny_masked(image, mask)
    gen     = torch.Generator(device=DEVICE).manual_seed(seed)
    vis_ref = extract_visible_patch(image, mask)

    call_kwargs = dict(
        prompt                      = prompt,
        negative_prompt             = NEG_PROMPT,
        image                       = image,
        control_image               = canny,
        mask_image                  = mask,
        num_inference_steps         = NUM_STEPS,
        guidance_scale              = GUIDANCE,
        controlnet_conditioning_scale = cn_scale,
        generator                   = gen,
    )

    if use_ip:
        pipe.set_ip_adapter_scale(ip_scale)
        call_kwargs["ip_adapter_image"] = vis_ref

    with torch.autocast("cuda"):
        result = pipe(**call_kwargs).images[0]

    torch.cuda.empty_cache()
    return result


print("Helpers defined: extract_canny_masked / extract_visible_patch / inpaint")

In [ ]:
# ─────────────────────────────────────────────
# CELL 5 — Visualize input pipeline (4 samples)
# ─────────────────────────────────────────────
import matplotlib.pyplot as plt

samples_vis = meta.sample(4, random_state=SEED).reset_index(drop=True)
n_cols = 5 if use_ip else 4
fig, axes = plt.subplots(4, n_cols, figsize=(4 * n_cols, 14))

for i, row in samples_vis.iterrows():
    occ = Image.open(OCC_DIR  / row["x_occ"]).convert("RGB").resize((512, 512))
    msk = Image.open(MASK_DIR / row["mask"]).convert("L").resize((512, 512))
    gt  = Image.open(GT_DIR   / row["x_gt"]).convert("RGB").resize((512, 512))
    can = extract_canny_masked(occ, msk)

    col = 0
    def show(ax, img, title, cmap=None):
        ax.imshow(img, cmap=cmap)
        if i == 0: ax.set_title(title, fontsize=9, fontweight="bold")
        ax.axis("off")

    show(axes[i, col], occ,              "x_occ");         col += 1
    show(axes[i, col], msk,              "Mask", "gray");  col += 1
    show(axes[i, col], can,              "Canny (masked)");col += 1
    if use_ip:
        vis = extract_visible_patch(occ, msk)
        show(axes[i, col], vis, "Visible ref"); col += 1
    show(axes[i, col], gt,               "x_gt (GT)")

    axes[i, 0].set_ylabel(f"bin={row.get('bin_label','?')}  occ={row['occlusion_ratio']:.2f}",
                          fontsize=8)

plt.suptitle("Input pipeline preview — SD1.5 + ControlNet + IP-Adapter", y=1.01)
plt.tight_layout()
plt.show()

In [ ]:
# ─────────────────────────────────────────────
# CELL 6 — Grid search cn_scale × ip_scale
# Dùng GRID_SAMPLE=40 ảnh, chọn combo (cn_scale, ip_scale) tốt nhất theo LPIPS
# ─────────────────────────────────────────────
from tqdm.auto import tqdm
from skimage.metrics import structural_similarity as ssim_fn, peak_signal_noise_ratio as psnr_fn
import lpips as lpips_lib

lpips_model = lpips_lib.LPIPS(net="alex").to(DEVICE)
lpips_model.eval()


# ── Metrics matching paper (Yan et al., ICCV 2019) ────────────────────────

def pixel_l1(gt: np.ndarray, pred: np.ndarray, mask01: np.ndarray) -> float:
    """Per-pixel L1 error, normalized [0,1], computed on masked region.
    Matches the paper's 'per-pixel L1 error' metric."""
    gt_f = gt.astype(np.float64) / 255.0
    pr_f = pred.astype(np.float64) / 255.0
    m    = mask01.astype(bool)
    if m.sum() == 0:
        return float(np.mean(np.abs(gt_f - pr_f)))
    return float(np.mean(np.abs(gt_f[m] - pr_f[m])))


def pixel_l2(gt: np.ndarray, pred: np.ndarray, mask01: np.ndarray) -> float:
    """Per-pixel L2 error (MSE), normalized [0,1], on masked region.
    Matches the paper's 'per-pixel L2 error' metric."""
    gt_f = gt.astype(np.float64) / 255.0
    pr_f = pred.astype(np.float64) / 255.0
    m    = mask01.astype(bool)
    if m.sum() == 0:
        return float(np.mean((gt_f - pr_f) ** 2))
    return float(np.mean((gt_f[m] - pr_f[m]) ** 2))


# ── Standard reconstruction metrics ──────────────────────────────────────

def masked_psnr(gt: np.ndarray, pred: np.ndarray, mask01: np.ndarray) -> float:
    """PSNR chỉ trên vùng mask (inpainted region)."""
    m   = mask01.astype(bool)
    if m.sum() == 0:
        return float(psnr_fn(gt, pred, data_range=255))
    gt_m  = gt[m].astype(np.float64)
    pr_m  = pred[m].astype(np.float64)
    mse   = np.mean((gt_m - pr_m) ** 2)
    if mse == 0:
        return 100.0
    return float(10 * np.log10(255.0 ** 2 / mse))


def masked_ssim(gt: np.ndarray, pred: np.ndarray, mask01: np.ndarray) -> float:
    gt_f = gt.astype(np.float32) / 255.0
    pr_f = pred.astype(np.float32) / 255.0
    _, ssim_map = ssim_fn(gt_f, pr_f, channel_axis=2, data_range=1.0, full=True)
    m = mask01.astype(bool)
    return float(ssim_map[m].mean()) if m.sum() > 0 else float(ssim_map.mean())


def masked_lpips(gt: np.ndarray, pred: np.ndarray, mask01: np.ndarray) -> float:
    m    = mask01.astype(np.float32)[..., None]
    gt_m = (gt.astype(np.float32) * m).astype(np.uint8)
    pr_m = (pred.astype(np.float32) * m).astype(np.uint8)
    gt_t = torch.from_numpy(gt_m).permute(2, 0, 1).unsqueeze(0).float() / 127.5 - 1.0
    pr_t = torch.from_numpy(pr_m).permute(2, 0, 1).unsqueeze(0).float() / 127.5 - 1.0
    with torch.no_grad():
        return float(lpips_model(gt_t.to(DEVICE), pr_t.to(DEVICE)).item())


grid_df_sample = meta.sample(GRID_SAMPLE, random_state=SEED).reset_index(drop=True)
ip_values      = IP_SCALE_GRID if use_ip else [0.0]
grid_res       = []

for cn_s in CN_SCALE_GRID:
    for ip_s in ip_values:
        ss, lp, ps = [], [], []
        tag = f"cn={cn_s} ip={ip_s}"
        for _, r in tqdm(grid_df_sample.iterrows(), total=len(grid_df_sample), desc=tag):
            occ  = Image.open(OCC_DIR  / r["x_occ"]).convert("RGB").resize((512, 512))
            msk  = Image.open(MASK_DIR / r["mask"]).convert("L").resize((512, 512))
            gt   = Image.open(GT_DIR   / r["x_gt"]).convert("RGB").resize((512, 512))
            pred = inpaint(occ, msk, PROMPT, cn_scale=cn_s, ip_scale=ip_s, seed=SEED)

            gt_np   = np.array(gt)
            pr_np   = np.array(pred)
            mk_np   = np.array(msk)
            mk01    = (mk_np > 127).astype(np.uint8)

            ss.append(masked_ssim(gt_np, pr_np, mk01))
            lp.append(masked_lpips(gt_np, pr_np, mk01))
            ps.append(masked_psnr(gt_np, pr_np, mk01))

        row_res = {
            "cn_scale": cn_s, "ip_scale": ip_s,
            "ssim_mean": float(np.mean(ss)),
            "lpips_mean": float(np.mean(lp)),
            "psnr_mean": float(np.mean(ps)),
        }
        grid_res.append(row_res)
        print(f"  {tag} | PSNR={row_res['psnr_mean']:.2f} SSIM={row_res['ssim_mean']:.4f} LPIPS={row_res['lpips_mean']:.4f}")

grid_df = pd.DataFrame(grid_res)
best_idx = int(grid_df["lpips_mean"].idxmin())
BEST_CN  = float(grid_df.loc[best_idx, "cn_scale"])
BEST_IP  = float(grid_df.loc[best_idx, "ip_scale"])
print(f"\nBest combo: cn_scale={BEST_CN}  ip_scale={BEST_IP}")
grid_df.to_csv(REPORT_DIR / "grid_search.csv", index=False)
print(grid_df.to_string(index=False))

In [ ]:
# ─────────────────────────────────────────────
# CELL 7 — Full inference 2000 ảnh (best cn_scale, ip_scale)
# ─────────────────────────────────────────────
import time
from tqdm.auto import tqdm

# Lấy đúng TEST_SIZE ảnh, stratified theo bin để đánh giá cân bằng
bin_dfs = []
per_bin = TEST_SIZE // len(BIN_EDGES)
for (lo, hi) in BIN_EDGES:
    subset = meta[(meta["occlusion_ratio"] >= lo) & (meta["occlusion_ratio"] < hi)]
    n = min(per_bin, len(subset))
    bin_dfs.append(subset.sample(n, random_state=SEED))
test_df = pd.concat(bin_dfs).sample(frac=1, random_state=SEED).reset_index(drop=True)
print(f"Test set: {len(test_df):,} images (≈{per_bin}/bin × {len(BIN_EDGES)} bins)")

pred_records = []
t0 = time.time()

for idx, r in tqdm(test_df.iterrows(), total=len(test_df), desc="Inference"):
    occ  = Image.open(OCC_DIR  / r["x_occ"]).convert("RGB").resize((512, 512))
    msk  = Image.open(MASK_DIR / r["mask"]).convert("L").resize((512, 512))
    gt   = Image.open(GT_DIR   / r["x_gt"]).convert("RGB").resize((512, 512))

    pred = inpaint(occ, msk, PROMPT, cn_scale=BEST_CN, ip_scale=BEST_IP, seed=SEED)

    pred_name = f"{r['stem']}.png"
    pred.save(PRED_DIR      / pred_name)
    gt.save  (EVAL_GT_DIR   / pred_name)
    pred.save(EVAL_PRED_DIR / pred_name)

    pred_records.append({
        "stem"            : r["stem"],
        "x_gt"            : r["x_gt"],
        "x_occ"           : r["x_occ"],
        "mask"            : r["mask"],
        "occlusion_ratio" : r["occlusion_ratio"],
        "bin_label"       : r.get("bin_label", "?"),
        "pred"            : pred_name,
    })

elapsed = time.time() - t0
pred_df = pd.DataFrame(pred_records)
print(f"\nInference done: {len(pred_df):,} images in {elapsed/60:.1f} min")
print(f"Config: cn_scale={BEST_CN}, ip_scale={BEST_IP}")
pred_df.to_csv(REPORT_DIR / "pred_index.csv", index=False)

In [ ]:
# ─────────────────────────────────────────────
# CELL 8 — Compute ALL metrics
#   Paper metrics  : L1, L2, ICP (Inception Conditional Probability), SS (Segmentation Score)
#   Modern metrics : PSNR, SSIM, LPIPS, FID
# ─────────────────────────────────────────────
from cleanfid import fid as cleanfid
import cv2
from torchvision import transforms
from torchvision.models import inception_v3
from torchvision.models.segmentation import deeplabv3_resnet101

# ── Load ICP model: Inception V3 pretrained on ImageNet ──────────────────
print("Loading Inception V3 (ICP) ...")
try:
    from torchvision.models import Inception_V3_Weights
    inception_net = inception_v3(weights=Inception_V3_Weights.IMAGENET1K_V1)
except (ImportError, AttributeError):
    inception_net = inception_v3(pretrained=True)   # torchvision < 0.13
inception_net.eval().to(DEVICE)

# ImageNet classes for car/vehicle (subset covering passenger cars, sports cars, etc.)
IMAGENET_CAR_CLASSES = [
    407,   # ambulance
    436,   # beach wagon / estate car
    511,   # grille / radiator grille
    627,   # limousine
    656,   # minivan
    705,   # passenger car / motorcar
    717,   # pickup truck
    734,   # police van
    751,   # racer / race car
    779,   # school bus
    817,   # sports car
    820,   # station wagon
    868,   # tow truck
]

inception_tf = transforms.Compose([
    transforms.Resize(299),
    transforms.CenterCrop(299),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]),
])

def compute_icp(pred_pil: Image.Image, mask_np_gray: np.ndarray) -> float:
    """
    Inception Conditional Probability (ICP) — Yan et al., ICCV 2019.
    Crops the recovered vehicle region (bbox of mask), passes through Inception V3,
    returns sum of softmax probabilities for ImageNet car-related classes.
    Higher = looks more like a real car to Inception.
    """
    ys, xs = np.where(mask_np_gray > 127)
    if len(ys) == 0:
        return 0.0
    pad = 16
    h, w = mask_np_gray.shape
    y0 = max(0, int(ys.min()) - pad)
    y1 = min(h, int(ys.max()) + pad)
    x0 = max(0, int(xs.min()) - pad)
    x1 = min(w, int(xs.max()) + pad)
    crop = pred_pil.crop((x0, y0, x1, y1))
    if crop.width < 10 or crop.height < 10:
        return 0.0
    inp = inception_tf(crop.convert("RGB")).unsqueeze(0).to(DEVICE)
    with torch.no_grad():
        out    = inception_net(inp)
        logits = out.logits if hasattr(out, "logits") else out
        probs  = torch.softmax(logits, dim=1)[0].cpu()
    return float(probs[IMAGENET_CAR_CLASSES].sum())


# ── Load SS model: DeepLab V3 (COCO/VOC 21-class, car = class 7) ─────────
print("Loading DeepLab V3 ResNet-101 (SS) ...")
deeplab_net = deeplabv3_resnet101(pretrained=True)
deeplab_net.eval().to(DEVICE)

#  VOC 21-class labels: 0=background, 7=car, 15=person, ...
CAR_CLASS_IDX = 7

seg_tf = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]),
])

def compute_ss(pred_pil: Image.Image, mask01: np.ndarray) -> float:
    """
    Segmentation Score (SS) — Yan et al., ICCV 2019.
    Runs pretrained DeepLab V3 on the recovered image.
    Measures the fraction of masked (inpainted) pixels that are classified
    as 'car' (VOC class 7) by DeepLab. Proxy for realism:
    if recovered car looks real, segmentation model should detect it as car.
    """
    inp = seg_tf(pred_pil.convert("RGB")).unsqueeze(0).to(DEVICE)
    with torch.no_grad():
        seg_out = deeplab_net(inp)["out"][0]          # (21, H, W)
    pred_seg = seg_out.argmax(0).cpu().numpy().astype(np.uint8)  # (H, W)
    pred_seg = cv2.resize(pred_seg, (512, 512), interpolation=cv2.INTER_NEAREST)

    m = mask01.astype(bool)
    if m.sum() == 0:
        return 0.0
    return float((pred_seg[m] == CAR_CLASS_IDX).mean())


print("ICP + SS models ready\n")

# ── Compute per-image metrics ─────────────────────────────────────────────
metric_rows = []
for _, r in tqdm(pred_df.iterrows(), total=len(pred_df), desc="Metrics"):
    gt_bgr = cv2.imread(str(GT_DIR   / r["x_gt"]))
    pr_bgr = cv2.imread(str(PRED_DIR / r["pred"]))
    if gt_bgr is None or pr_bgr is None:
        continue
    gt_np  = cv2.resize(cv2.cvtColor(gt_bgr, cv2.COLOR_BGR2RGB), (512, 512))
    pr_np  = cv2.resize(cv2.cvtColor(pr_bgr, cv2.COLOR_BGR2RGB), (512, 512))
    mk_raw = cv2.imread(str(MASK_DIR / r["mask"]), cv2.IMREAD_GRAYSCALE)
    mk_np  = cv2.resize(mk_raw, (512, 512))
    mk01   = (mk_np > 127).astype(np.uint8)

    pr_pil = Image.fromarray(pr_np)

    metric_rows.append({
        "stem"            : r["stem"],
        "occlusion_ratio" : r["occlusion_ratio"],
        "bin_label"       : r.get("bin_label", "?"),
        # Paper metrics
        "l1"   : pixel_l1(gt_np, pr_np, mk01),
        "l2"   : pixel_l2(gt_np, pr_np, mk01),
        "icp"  : compute_icp(pr_pil, mk_np),
        "ss"   : compute_ss(pr_pil, mk01),
        # Modern metrics
        "psnr" : masked_psnr(gt_np, pr_np, mk01),
        "ssim" : masked_ssim(gt_np, pr_np, mk01),
        "lpips": masked_lpips(gt_np, pr_np, mk01),
    })

metric_df = pd.DataFrame(metric_rows)

print("Computing FID ...")
fid_value = cleanfid.compute_fid(str(EVAL_GT_DIR), str(EVAL_PRED_DIR), mode="clean")
print(f"FID = {fid_value:.2f}")

metric_df.to_csv(REPORT_DIR / "metrics_per_image.csv", index=False)
print(f"\nSaved: {REPORT_DIR}/metrics_per_image.csv")
print(f"\nOverall stats ({len(metric_df):,} images):")
print(metric_df[["l1","l2","icp","ss","psnr","ssim","lpips"]].describe().round(4))

In [ ]:
# ─────────────────────────────────────────────
# CELL 9 — Report: overall + per-bin + so sánh với paper (Yan et al., ICCV 2019)
# ─────────────────────────────────────────────
import json

ALL_METRICS = ["l1", "l2", "icp", "ss", "psnr", "ssim", "lpips"]

def bin_summary(df: pd.DataFrame, lo: float, hi: float, label: str) -> dict:
    sub = df[(df["occlusion_ratio"] >= lo) & (df["occlusion_ratio"] < hi)]
    if len(sub) == 0:
        return {"bin": label, "n": 0, **{m: None for m in ALL_METRICS}}
    row = {"bin": label, "n": len(sub)}
    for m in ALL_METRICS:
        if m in sub.columns:
            row[m] = round(float(sub[m].mean()), 4)
    return row

overall = {"bin": "overall", "n": len(metric_df)}
for m in ALL_METRICS:
    if m in metric_df.columns:
        overall[m] = round(float(metric_df[m].mean()), 4)

rows = [overall]
for (lo, hi), label in zip(BIN_EDGES, BIN_LABELS):
    rows.append(bin_summary(metric_df, lo, hi, label))

summary_df = pd.DataFrame(rows)
summary_df.insert(summary_df.columns.get_loc("psnr") + 3, "fid",
                  [round(fid_value, 2)] + [None] * len(BIN_EDGES))

# ── Print our results ─────────────────────────────────────────────────────
print("=" * 80)
print("EVALUATION RESULTS — SD1.5 + ControlNet Canny + IP-Adapter")
print(f"Config: cn_scale={BEST_CN}  ip_scale={BEST_IP}  steps={NUM_STEPS}  n={len(metric_df):,}")
print("=" * 80)
print(summary_df.to_string(index=False))
print("=" * 80)

# ── Paper comparison table (Yan et al., ICCV 2019 — Table 4, Synthetic M^gt) ──
print("\n" + "=" * 80)
print("COMPARISON WITH Yan et al. (ICCV 2019) — Appearance Recovery (Synthetic, M^gt)")
print("Note: Paper uses predicted GT masks; our setup uses synthetic ground-truth masks.")
print("      ICP/SS computed differently (paper: Cityscapes DeepLab; ours: COCO DeepLab V3)")
print("=" * 80)

paper_rows = [
    {"method": "Deepfill [50]",       "l1": 0.0284, "l2": 0.0107, "icp": 0.5620, "ss": 0.8295},
    {"method": "Liu et al. [27]",     "l1": 0.0272, "l2": 0.0074, "icp": 0.6284, "ss": 0.8672},
    {"method": "Pathak et al. [35]",  "l1": 0.0207, "l2": 0.0088, "icp": 0.5708, "ss": 0.8517},
    {"method": "pix2pix [20]",        "l1": 0.0174, "l2": 0.0060, "icp": 0.7081, "ss": 0.9410},
    {"method": "SeGAN [12]",          "l1": 0.0181, "l2": 0.0055, "icp": 0.6662, "ss": 0.9371},
    {"method": "Yan et al. 1st iter.","l1": 0.0159, "l2": 0.0038, "icp": 0.7436, "ss": 0.9458},
    {"method": "Yan et al. 2nd iter.","l1": 0.0158, "l2": 0.0039, "icp": 0.7267, "ss": 0.9447},
    {"method": ">>> Ours (SD1.5+CN+IPA)",
     "l1" : overall.get("l1"),
     "l2" : overall.get("l2"),
     "icp": overall.get("icp"),
     "ss" : overall.get("ss"),
    },
]
paper_df = pd.DataFrame(paper_rows)
pd.set_option("display.float_format", "{:.4f}".format)
print(paper_df.to_string(index=False))
print("=" * 80)
print("↓ = lower is better  |  ↑ = higher is better")
print("L1 ↓   L2 ↓   ICP ↑   SS ↑")

# ── Save ─────────────────────────────────────────────────────────────────
summary_df.to_csv(REPORT_DIR / "summary_table.csv", index=False)
paper_df.to_csv(REPORT_DIR / "paper_comparison.csv", index=False)

meta_info = {
    "config"         : "SD1.5 + ControlNet Canny + IP-Adapter",
    "sd_model"       : SD_MODEL_ID,
    "cn_model"       : CN_MODEL_ID,
    "ip_weight"      : IP_WEIGHT,
    "scheduler"      : pipe.scheduler.__class__.__name__,
    "num_steps"      : NUM_STEPS,
    "guidance_scale" : GUIDANCE,
    "cn_scale"       : BEST_CN,
    "ip_scale"       : BEST_IP,
    "prompt"         : PROMPT,
    "neg_prompt"     : NEG_PROMPT,
    "test_size"      : int(len(metric_df)),
    "overall_l1"     : overall.get("l1"),
    "overall_l2"     : overall.get("l2"),
    "overall_icp"    : overall.get("icp"),
    "overall_ss"     : overall.get("ss"),
    "overall_psnr"   : overall.get("psnr"),
    "overall_ssim"   : overall.get("ssim"),
    "overall_lpips"  : overall.get("lpips"),
    "fid"            : round(fid_value, 2),
    "xformers"       : xformers_ok,
    "ip_adapter"     : use_ip,
    "paper_ref"      : "Yan et al., ICCV 2019 — Visualizing the Invisible",
    "paper_best_l1"  : 0.0158,
    "paper_best_l2"  : 0.0038,
    "paper_best_icp" : 0.7436,
    "paper_best_ss"  : 0.9458,
}
with open(REPORT_DIR / "eval_config.json", "w") as f:
    json.dump(meta_info, f, indent=2)

print("\nFiles saved:")
print(f"  {REPORT_DIR}/summary_table.csv")
print(f"  {REPORT_DIR}/paper_comparison.csv")
print(f"  {REPORT_DIR}/metrics_per_image.csv")
print(f"  {REPORT_DIR}/eval_config.json")

In [ ]:
# ─────────────────────────────────────────────
# CELL 10 — Visualization: metric by bin + best/worst cases
# ─────────────────────────────────────────────
import matplotlib.pyplot as plt
import matplotlib.ticker as mtick

# ── 10a. Bar charts: PSNR / SSIM / LPIPS theo bin ───────────────────────
plot_df = summary_df[summary_df["bin"] != "overall"].copy()
fig, axes = plt.subplots(1, 3, figsize=(14, 4))

for ax, col, label, ascending in [
    (axes[0], "psnr",  "PSNR ↑ (dB)",    False),
    (axes[1], "ssim",  "SSIM ↑",          False),
    (axes[2], "lpips", "LPIPS ↓",         True),
]:
    colors = ["#3498db", "#2ecc71", "#e74c3c"]
    bars = ax.bar(plot_df["bin"], plot_df[col], color=colors, edgecolor="white", width=0.5)
    ax.set_title(label, fontsize=11, fontweight="bold")
    ax.set_xlabel("Occlusion level")
    for bar, v in zip(bars, plot_df[col]):
        ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.002,
                f"{v:.3f}", ha="center", va="bottom", fontsize=9)

plt.suptitle("Metric by occlusion bin — SD1.5 + ControlNet Canny + IP-Adapter", y=1.02)
plt.tight_layout()
plt.show()

# ── 10b. PSNR scatter: ratio vs psnr ─────────────────────────────────────
fig2, ax2 = plt.subplots(figsize=(8, 4))
sc = ax2.scatter(metric_df["occlusion_ratio"], metric_df["psnr"],
                  c=metric_df["psnr"], cmap="RdYlGn", alpha=0.4, s=8)
plt.colorbar(sc, ax=ax2, label="PSNR (dB)")
for (lo, hi) in BIN_EDGES:
    ax2.axvline(lo, color="gray", linestyle="--", linewidth=0.8)
ax2.axvline(BIN_EDGES[-1][1], color="gray", linestyle="--", linewidth=0.8)
ax2.set_xlabel("Occlusion ratio")
ax2.set_ylabel("PSNR (dB)")
ax2.set_title("PSNR vs occlusion ratio per image")
plt.tight_layout()
plt.show()

# ── 10c. Grid search heatmap (nếu dùng IP-Adapter) ───────────────────────
if use_ip and len(grid_df) > 1:
    pivot_ssim  = grid_df.pivot(index="ip_scale", columns="cn_scale", values="ssim_mean")
    pivot_lpips = grid_df.pivot(index="ip_scale", columns="cn_scale", values="lpips_mean")
    fig3, (ax3a, ax3b) = plt.subplots(1, 2, figsize=(12, 4))
    import matplotlib
    im1 = ax3a.imshow(pivot_ssim.values,  cmap="Blues",   aspect="auto")
    im2 = ax3b.imshow(pivot_lpips.values, cmap="Oranges_r", aspect="auto")
    for ax_h, pivot, title in [(ax3a, pivot_ssim, "SSIM ↑"), (ax3b, pivot_lpips, "LPIPS ↓")]:
        ax_h.set_xticks(range(len(pivot.columns)))
        ax_h.set_xticklabels([str(c) for c in pivot.columns])
        ax_h.set_yticks(range(len(pivot.index)))
        ax_h.set_yticklabels([str(r) for r in pivot.index])
        ax_h.set_xlabel("cn_scale")
        ax_h.set_ylabel("ip_scale")
        ax_h.set_title(f"Grid search — {title}")
    plt.colorbar(im1, ax=ax3a)
    plt.colorbar(im2, ax=ax3b)
    plt.tight_layout()
    plt.show()

# ── 10d. Visual grid: 2 mẫu / bin (best + worst theo PSNR) ──────────────
fig4, axes4 = plt.subplots(len(BIN_EDGES) * 2, 5, figsize=(18, 5 * len(BIN_EDGES) * 2))
row_idx = 0
for (lo, hi), label in zip(BIN_EDGES, BIN_LABELS):
    bin_m = metric_df[(metric_df["occlusion_ratio"] >= lo) &
                      (metric_df["occlusion_ratio"] <  hi)]
    if len(bin_m) == 0:
        continue
    best_stem  = bin_m.loc[bin_m["psnr"].idxmax(),  "stem"]
    worst_stem = bin_m.loc[bin_m["psnr"].idxmin(), "stem"]
    for stem, tag in [(best_stem, "BEST"), (worst_stem, "WORST")]:
        row_m = pred_df[pred_df["stem"] == stem].iloc[0]
        m_row = bin_m[bin_m["stem"] == stem].iloc[0]
        imgs = [
            (Image.open(GT_DIR   / row_m["x_gt"]).convert("RGB"),  "x_gt"),
            (Image.open(OCC_DIR  / row_m["x_occ"]).convert("RGB"), "x_occ"),
            (Image.open(MASK_DIR / row_m["mask"]).convert("L"),    "Mask"),
            (extract_canny_masked(
                Image.open(OCC_DIR / row_m["x_occ"]).convert("RGB"),
                Image.open(MASK_DIR / row_m["mask"]).convert("L")
            ),                                                      "Canny"),
            (Image.open(PRED_DIR / row_m["pred"]).convert("RGB"),  "x_hat"),
        ]
        for col_j, (img, title) in enumerate(imgs):
            ax = axes4[row_idx, col_j]
            ax.imshow(img, cmap="gray" if title == "Mask" else None)
            if row_idx == 0: ax.set_title(title, fontsize=9, fontweight="bold")
            ax.axis("off")
        psnr_v = m_row["psnr"]; ssim_v = m_row["ssim"]; lpips_v = m_row["lpips"]
        axes4[row_idx, 0].set_ylabel(
            f"Bin {label}\n{tag}\nPSNR={psnr_v:.1f}  SSIM={ssim_v:.3f}\nLPIPS={lpips_v:.3f}",
            fontsize=7, labelpad=4
        )
        row_idx += 1

# Ẩn axes còn dư
for ax in axes4[row_idx:].flatten():
    ax.axis("off")

plt.suptitle("Best & Worst per bin — SD1.5 + ControlNet + IP-Adapter", y=1.01)
plt.tight_layout()
plt.show()

In [ ]:
# ─────────────────────────────────────────────
# CELL 11 — VRAM report + đóng gói reports.zip
# ─────────────────────────────────────────────
import zipfile, subprocess

# VRAM
if torch.cuda.is_available():
    peak_mb = torch.cuda.max_memory_allocated() / 1024**2
    props   = torch.cuda.get_device_properties(0)
    print(f"GPU         : {props.name}")
    print(f"Total VRAM  : {props.total_memory/1024**3:.1f} GiB")
    print(f"Peak alloc  : {peak_mb:.0f} MiB ({peak_mb/1024:.2f} GiB)")
    print(f"xFormers    : {xformers_ok}  |  IP-Adapter: {use_ip}")
    try:
        smi = subprocess.check_output(
            ["nvidia-smi","--query-gpu=name,memory.total,memory.used,memory.free",
             "--format=csv,noheader,nounits"],
            stderr=subprocess.DEVNULL,
        ).decode().strip()
        print("nvidia-smi  :", smi)
    except Exception:
        pass

# Zip reports
zip_path = Path("/kaggle/working/eval_cn_ipa_reports.zip")
report_files = list(REPORT_DIR.glob("*"))
with zipfile.ZipFile(zip_path, "w", zipfile.ZIP_DEFLATED, compresslevel=3) as zf:
    for f in sorted(report_files):
        zf.write(f, f"reports/{f.name}")
print(f"\nDownload  : {zip_path.name}  ({zip_path.stat().st_size/1e6:.1f} MB)")
print("Contains  :")
for f in sorted(report_files):
    print(f"  reports/{f.name}")